# 03: Interactive Composite 3D Mesh with Vector Overlays (GeoJSON)

This notebook demonstrates **interactive multi-layer 3D geospatial visualization** combining a spherical terrain mesh with vector lines:
1. Loading vector GeoJSON files representing global transportation corridors:
   - International Flight Corridors (`data/vector/flight_routes.geojson`)
   - Maritime Shipping Lanes (`data/vector/shipping_routes.geojson`)
2. Styling vector layers with `VectorLayerConfig` (colors, linewidths, opacity, and z-order layering).
3. Interactively rotating and exploring the composite 3D globe with `animate_mesh_with_vector_files(..., sFilename_out=None)`.

In [ ]:
import json
import os
from pathlib import Path
import numpy as np
import geovista as gv

import pyvista as pv

# Configure interactive 3D rendering backend in Jupyter:
# - If trame is installed (pip install "pyvista[jupyter]"), embeds an interactive 3D WebGL widget inside the cell.
# - If trame is not installed, opens a standalone interactive desktop window on macOS.
try:
    import trame

    pv.set_jupyter_backend("client")
    print("✓ Interactive backend: in-cell 3D WebGL widget (trame)")
except ImportError:
    pv.set_jupyter_backend("none")
    print("ℹ Interactive backend: native desktop 3D window")
    print("  To embed interactive 3D widgets directly inside notebook cells: pip install 'pyvista[jupyter]'")

import pyearthviz3d
from pyearthviz3d import (
    VectorLayerConfig,
    VisualizationConfig,
    animate_mesh_with_vector_files,
)

# Resolve repository root and data paths
REPO_ROOT = Path(os.getcwd()).resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

FLIGHTS_GEOJSON = REPO_ROOT / "data" / "vector" / "flight_routes.geojson"
SHIPPING_GEOJSON = REPO_ROOT / "data" / "vector" / "shipping_routes.geojson"

print(f"Flight routes file:   {FLIGHTS_GEOJSON} (exists: {FLIGHTS_GEOJSON.exists()})")
print(f"Shipping routes file: {SHIPPING_GEOJSON} (exists: {SHIPPING_GEOJSON.exists()})")

## Step 1: Inspect Vector GIS Features

In [ ]:
with open(FLIGHTS_GEOJSON, "r") as f:
    flights_data = json.load(f)

with open(SHIPPING_GEOJSON, "r") as f:
    shipping_data = json.load(f)

print(f"Flight routes:   {len(flights_data['features'])} line features")
for feat in flights_data["features"][:3]:
    props = feat["properties"]
    print(f"  • {props['name']} ({props['route_id']}) - {props['category']}")

print(f"\nShipping routes: {len(shipping_data['features'])} line features")
for feat in shipping_data["features"][:3]:
    props = feat["properties"]
    print(f"  • {props['corridor']} (Density: {props['density']}) - Chokepoints: {props['chokepoints']}")

## Step 2: Construct Base Spherical Topography Mesh

In [ ]:
lons = np.linspace(-180, 180, 73)
lats = np.linspace(-90, 90, 37)
mesh = gv.Transform.from_1d(lons, lats)

lon_grid, lat_grid = np.meshgrid(
    np.radians((lons[:-1] + lons[1:]) / 2.0),
    np.radians((lats[:-1] + lats[1:]) / 2.0),
)
topo = np.sin(2.0 * lat_grid) * np.cos(2.0 * lon_grid) * 3000.0
mesh.cell_data["elevation"] = topo.ravel()
valid_indices = np.arange(mesh.n_cells)

print(f"Base mesh ready: {mesh.n_cells} cells, elevation range [{topo.min():.0f}, {topo.max():.0f}] m.")

## Step 3: Configure Layer Styling and Hierarchy

We use `VectorLayerConfig` to define styling properties for each vector layer:
- **Flight routes**: Rendered in crimson with line width 3.0 and high z-order priority (`z_order=10`).
- **Shipping lanes**: Rendered in gold with line width 2.5 and lower z-order (`z_order=5`).

In [ ]:
flight_layer_config = VectorLayerConfig(
    color="crimson",
    linewidth=3.0,
    opacity=1.0,
    layer_name="Commercial Flight Corridors",
    z_order=10,
)

shipping_layer_config = VectorLayerConfig(
    color="gold",
    linewidth=2.5,
    opacity=0.9,
    layer_name="Global Shipping Corridors",
    z_order=5,
)

vector_configs = [flight_layer_config, shipping_layer_config]
vector_files = [str(FLIGHTS_GEOJSON), str(SHIPPING_GEOJSON)]

## Step 4: Interactive 3D Composite Visualization

Passing `sFilename_out=None` launches the interactive 3D view.
You can rotate the globe to track trans-Pacific and trans-Atlantic flight corridors as well as maritime passages through the Suez and Panama canals.

In [ ]:
viz_config = VisualizationConfig(
    longitude_focus=-40.0,  # Vantage centered over the Atlantic Ocean
    latitude_focus=30.0,
    zoom_factor=0.85,
    show_coastlines=True,
    show_graticule=True,
    coastline_color="white",
    coastline_width=1.0,
    colormap="terrain",
    verbose=True,
)

# Launch interactive 3D view
result = animate_mesh_with_vector_files(
    pMesh=mesh,
    aValid_cell_indices=valid_indices,
    pConfig=viz_config,
    sScalar="elevation",
    sUnit="m",
    aFilename_vector_in=vector_files,
    aVector_config=vector_configs,
    sFilename_out=None,  # Interactive 3D mode
    return_detailed_result=True,
)

if hasattr(result, "get_summary"):
    print(result.get_summary())

## Optional: Static PNG Export

In [ ]:
from IPython.display import Image, display

output_png = "composite_mesh_vectors_export.png"
animate_mesh_with_vector_files(
    pMesh=mesh,
    aValid_cell_indices=valid_indices,
    pConfig=viz_config,
    sScalar="elevation",
    sUnit="m",
    aFilename_vector_in=vector_files,
    aVector_config=vector_configs,
    sFilename_out=output_png,
)
display(Image(output_png, width=600))